In [1]:
import psycopg2
import dotenv 
import os
from sqlalchemy import create_engine, text

In [3]:
# Connect to postgresql and create database engine
dotenv.load_dotenv("../backend/.env")

DB_HOST=os.getenv("DB_HOST")
DB_PORT=os.getenv("DB_PORT")
DB_NAME=os.getenv("DB_NAME")
DB_USER=os.getenv("DB_USER")
DB_PASSWORD=os.getenv("DB_PASSWORD")

engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Connection successful: ", result.fetchone())


In [4]:
# define and create the table 
create_table_query = text("""
CREATE TABLE IF NOT EXISTS transactions (
    id                  SERIAL PRIMARY KEY,
    transaction_type    TEXT NOT NULL,
    date                DATE NOT NULL,
    amount              FLOAT NOT NULL,
    description         TEXT NOT NULL,
    transaction_code    TEXT,
    category            TEXT,
    fingerprint         TEXT UNIQUE NOT NULL,
    created_at          TIMESTAMP DEFAULT NOW()
    );
""")

with engine.connect() as connection:
    connection.execute(create_table_query)
    connection.commit()
    print("table created successfully")

In [8]:
# test the connection ==> insert a row and read it back
import sys

sys.path.append("../backend")
from backend.parser import parse_bmo_csv

df = parse_bmo_csv("statement.csv")
records = df.to_dict(orient="records")

insert_query = text("""
INSERT INTO transactions
    (transaction_type, date, amount, description, transaction_code,fingerprint) 
VALUES 
    (:transaction_type, :date, :amount, :description, :transaction_code,:fingerprint)
ON CONFLICT (fingerprint) DO NOTHING;
""")

# Insert rows 
with engine.connect() as connection:
    for record in records:
        connection.execute(insert_query, record)
    connection.commit()
    print(f"Inserted {len(records)} rows successfully")
    
# Fetch rows to verify 
with engine.connect() as connection:
    result = connection.execute(text("SELECT * FROM transactions"))
    rows = result.fetchmany(10)
    for row in rows: 
        print(row)


In [10]:
# Counting the number of rows in the table
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM transactions"))
    count = result.fetchone()[0]
    print(count) 
